In [1]:
# Install required libraries:
# pypdf — extracts text from PDF files
# sentence-transformers — converts text to vectors for similarity search
# faiss-cpu — fast similarity search (finds most relevant chunks)
# transformers — loads and runs the LLM (Mistral)
# accelerate — helps load large models efficiently on GPU

!pip install transformers torch pandas pypdf sentence-transformers faiss-cpu accelerate -q

print("All libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 79.2 MB/s eta 0:00:00
All libraries installed


In [2]:
import re
from pypdf import PdfReader
from google.colab import files

# Upload the 4 Austrian law PDFs (EStG, KStG, UStG, GrEStG)
print("Upload your 4 PDF files")
uploaded = files.upload()

# Dictionary to store extracted text — key: filename, value: full text
law_texts = {}

for filename, content in uploaded.items():

    # Save the uploaded file to disk so pypdf can read it
    with open(filename, "wb") as f:
        f.write(content)

    # Open the PDF and prepare to read page by page
    reader = PdfReader(filename)
    full_text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        # Some pages may be empty or image-only so skip those
        if page_text:
            full_text += page_text + "\n"

    # Store the full text of this law
    law_texts[filename] = full_text
    print(f"Done: {filename} — {len(reader.pages)} pages, {len(full_text)} characters")

print(f"\nTotal laws loaded: {len(law_texts)}")

Please upload your 4 PDF files...


Saving EStG 1988, Fassung vom 03.04.2026-2.pdf to EStG 1988, Fassung vom 03.04.2026-2.pdf
Saving GrEStG 1987, Fassung vom 04.04.2026.pdf to GrEStG 1987, Fassung vom 04.04.2026.pdf
Saving KStG 1988, Fassung vom 03.04.2026-3.pdf to KStG 1988, Fassung vom 03.04.2026-3.pdf
Saving UStG 1994, Fassung vom 04.04.2026.pdf to UStG 1994, Fassung vom 04.04.2026.pdf
Done: EStG 1988, Fassung vom 03.04.2026-2.pdf — 228 pages, 950467 characters
Done: GrEStG 1987, Fassung vom 04.04.2026.pdf — 15 pages, 65134 characters
Done: KStG 1988, Fassung vom 03.04.2026-3.pdf — 53 pages, 214411 characters
Done: UStG 1994, Fassung vom 04.04.2026.pdf — 73 pages, 313304 characters

Total laws loaded: 4


In [3]:
# Split each law into smaller chunks at paragraph boundaries - §
# We can't feed the entire law to the model - context window is too small
# Each § is one legal rule so it's a natural and meaningful chunk size

def split_into_chunks(text, law_name, min_length=50, max_length=3000):
    chunks = []

    # Split at every § symbol — each paragraph starts here
    raw_splits = re.split(r'(?=§\s*\d)', text)

    for split in raw_splits:
        split = split.strip()

        # Skip very short chunks — usually just headers or noise
        if len(split) < min_length:
            continue

        # If chunk is too long then break it further at sentence boundaries
        if len(split) > max_length:
            sentences = split.split('. ')
            current_chunk = ""
            for sentence in sentences:
                if len(current_chunk) + len(sentence) < max_length:
                    current_chunk += sentence + ". "
                else:
                    if len(current_chunk) > min_length:
                        chunks.append({"text": current_chunk.strip(), "source": law_name})
                    current_chunk = sentence + ". "
            if len(current_chunk) > min_length:
                chunks.append({"text": current_chunk.strip(), "source": law_name})
        else:
            chunks.append({"text": split, "source": law_name})

    return chunks

# Process each law and collect all chunks into one list
all_chunks = []

for filename, text in law_texts.items():
    # Extract clean law name from filename (like 'EStG' from 'EStG 1988...')
    law_name = filename.split(' ')[0]

    chunks = split_into_chunks(text, law_name)
    all_chunks.extend(chunks)
    print(f"{law_name}: {len(chunks)} chunks")

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"\nExample chunk:\n{all_chunks[10]['text'][:300]}")

EStG: 2684 chunks
GrEStG: 140 chunks
KStG: 612 chunks
UStG: 544 chunks

Total chunks: 3980

Example chunk:
§ 5. Gewinn der protokollierten Gewerbetreibenden (Anm.: Gewinn der 
rechnungslegungspflichtigen Gewerbetreibenden)


In [4]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

# Load a multilingual sentence transformer model
# 'paraphrase-multilingual-MiniLM-L12-v2' works well for German text
# It converts any text into a 384-dimensional vector
print("Loading embedding model...")
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Embedding model loaded!")

# Extract just the text from each chunk for embedding
chunk_texts = [chunk["text"] for chunk in all_chunks]

print(f"\nEmbedding {len(chunk_texts)} chunks...")

# Convert each chunk into a vector, similar chunks will have similar vectors
# This is what allows us to search for relevant chunks later
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True  # FAISS requires numpy arrays
)

print(f"\nEmbeddings shape: {chunk_embeddings.shape}")
# Expected: (3980, 384) — one 384-dimensional vector per chunk

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!

Embedding 3980 chunks...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]


Embeddings shape: (3980, 384)


In [5]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Function to find the most relevant law chunks for a given question
# It embeds the question, then compares it against all chunk embeddings
# and returns the top-k most similar chunks

def find_relevant_chunks(question, top_k=5):

    # Convert the question into a vector using the same embedding model
    question_embedding = embedding_model.encode([question], convert_to_numpy=True)

    # Compare the question vector against all 3980 chunk vectors
    # Result is a score between 0-1 for each chunk — higher means more relevant
    scores = cosine_similarity(question_embedding, chunk_embeddings)[0]

    # Get indices of the top-k highest scoring chunks
    top_indices = np.argsort(scores)[::-1][:top_k]

    # Return the actual chunk texts
    return [all_chunks[i]["text"] for i in top_indices]

# Quick test to verify retrieval works
test_question = "Wie hoch ist die Körperschaftsteuer?"
results = find_relevant_chunks(test_question)
print(f"Test question: {test_question}")
print(f"\nTop relevant chunk:\n{results[0][:300]}")

Test question: Wie hoch ist die Körperschaftsteuer?

Top relevant chunk:
§ 8 Abs. 4 Z 1 abzugsfähig sind. 
 6. Die Steuern vom Einkommen und sonstige Personensteuern und  die aus Anlass einer 
unentgeltlichen Grundstücksübertragung anfallende Grunderwerbsteuer, Eintragungsgebühren 
und andere Nebenkosten; weiters  die Umsatzsteuer, die auf nichtabzugsfähige Aufwendungen 


In [6]:
# Install Groq client library to use Groq API for LLM inference
!pip install groq -q

print("Groq installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 7.6 MB/s eta 0:00:00
Groq installed!


In [7]:
import pandas as pd
from groq import Groq
from google.colab import files

# Initialize Groq client with your new API key
client = Groq(api_key="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")

# Upload and load the questions dataset
print("Upload dataset_clean.csv")
files.upload()
questions_df = pd.read_csv("dataset_clean.csv")
print(f"Loaded {len(questions_df)} questions")

def answer_with_rag(question):
    # Retrieve top 3 most relevant law chunks for this question
    relevant_chunks = find_relevant_chunks(question, top_k=3)

    # Combine chunks into one context string
    context = "\n\n".join(relevant_chunks)

    # Build prompt with retrieved law context
    prompt = (
        "Du bist ein österreichischer Steuerrechtsexperte. "
        "Beantworte die folgende Frage präzise auf Deutsch, basierend auf den angegebenen Gesetzestexten "
        "(EStG, KStG, UStG, GrEStG). Nenne den relevanten Paragraphen wenn möglich.\n\n"
        f"Gesetzestexte:\n{context}\n\n"
        f"Frage: {question}\n\n"
        "Antwort:"
    )

    # Send to Groq API and get answer
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=250,
        temperature=0.1
    )

    return response.choices[0].message.content.strip()

# Test on one question first
test_q = questions_df["prompt"].iloc[0]
test_a = answer_with_rag(test_q)
print(f"Question: {test_q}")
print(f"\nAnswer: {test_a}")

Upload dataset_clean.csv


Saving dataset_clean.csv to dataset_clean.csv
Loaded 643 questions
Question: Was ist die steuerliche Bemessungsgrundlage für die Körperschaftsteuer?

Answer: Die steuerliche Bemessungsgrundlage für die Körperschaftsteuer ist gemäß § 4 Abs. 1 KStG der nach § 8 KStG ermittelte Körperschaftsteuerbesitzstand.


In [8]:
import time
import os

from google.colab import files
print("Upload model3_rag_results.csv...")
files.upload()

# Load existing results to resume if interrupted
if os.path.exists("model3_rag_results.csv"):
    existing_df = pd.read_csv("model3_rag_results.csv")
    # Only count questions that have a real non-empty answer
    answered_ids = set(existing_df[existing_df['answer'].notna() & (existing_df['answer'] != "")]["id"].tolist())
    all_answers = existing_df.to_dict('records')
    print(f"Resuming — {len(answered_ids)} already answered, {643 - len(answered_ids)} remaining")
else:
    answered_ids = set()
    all_answers = []
    print("Starting fresh")

for i, row in questions_df.iterrows():
    question_id = row["id"]
    question = row["prompt"]

    # Skip already answered questions
    if question_id in answered_ids:
        continue

    # Try up to 3 times in case of rate limit errors
    for attempt in range(3):
        try:
            answer = answer_with_rag(question)
            break  # Success — exit retry loop
        except Exception as e:
            if "429" in str(e):
                # Rate limit — wait 60 seconds before retrying
                print(f"Rate limited on {question_id} (attempt {attempt+1}/3), waiting 60s...")
                time.sleep(60)
            else:
                print(f"Error on {question_id}: {e}")
                answer = ""
                break
    else:
        # All 3 attempts failed — store empty and move on
        answer = ""

    all_answers.append({"id": question_id, "answer": answer})

    # Save after every single question in case of disconnect
    pd.DataFrame(all_answers).to_csv("model3_rag_results.csv", index=False)

    if (i + 1) % 50 == 0:
        print(f"Progress: {i + 1}/643")

    # Wait between requests to avoid rate limits
    time.sleep(2)

print("All done!")
print(pd.read_csv("model3_rag_results.csv").head(3))

Upload model3_rag_results.csv...


Saving model3_rag_results.csv to model3_rag_results.csv
Resuming — 413 already answered, 230 remaining
Progress: 450/643
Progress: 500/643
Progress: 550/643
Progress: 600/643
All done!
             id                                             answer
0  CORP-TAX-001  Die steuerliche Bemessungsgrundlage für die Kö...
1  CORP-TAX-002  Die Frage ist jedoch nicht direkt mit den ange...
2  CORP-TAX-003  Laut § 50 Abs. 2 Z 3 des Bewertungsgesetzes 19...


In [10]:
import pandas as pd

df = pd.read_csv("model3_rag_results.csv")

# Remove duplicates — keep the one with a non-empty answer
df = df.sort_values('answer', na_position='last')
df = df.drop_duplicates(subset='id', keep='first')
df = df.sort_index().reset_index(drop=True)

print(f"Rows after cleanup: {len(df)}")
empty = df['answer'].isna() | (df['answer'] == '')
print(f"Empty: {len(df[empty])}")

df.to_csv("model3_rag_results.csv", index=False)
print("Saved!")

Rows after cleanup: 643
Empty: 0
Saved!


In [11]:
from google.colab import files
files.download("model3_rag_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>